# Comparative Evaluation — CNN vs BiLSTM vs SVM

Loads results from the CNN and BiLSTM pipelines, trains an SVM baseline, and produces a final comparative evaluation across all three models.

## Prerequisites

**Dataset:** `TENSORS_DIR` must point to `/kaggle/input/pr-a2-preprocesseddataset`.

**Repository:** The repo must be cloned into `/kaggle/working/` so that `src/` modules are accessible:

```bash
git clone https://github.com/marticasasn/ELEC3612-Assignment2
```

**CNN and LSTM results** must be uploaded as separate Kaggle Datasets:
- `cnn-results` containing `best_cnn.pth` and `cnn_predictions.csv`
- `lstm-results` containing `best_lstm.pth` and `lstm_predictions.csv`

Run the clone cell below before executing any other cell.

In [ ]:
from pathlib import Path
REPO_DIR = Path("/kaggle/working/ELEC3612-Assignment2")
if not REPO_DIR.exists():
    !git clone https://github.com/marticasasn/ELEC3612-Assignment2 {REPO_DIR}
else:
    print("Repository already exists, skipping clone.")

## 1. Imports and Configuration

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sys.path.insert(0, str(REPO_DIR))

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src.dataset import MFCCDataset, SpectrogramDataset
from src.evaluate import compute_metrics, plot_confusion_matrix, plot_training_curves

# --- Paths ---
TENSORS_DIR      = Path("/kaggle/input/pr-a2-preprocesseddataset")
CNN_CHECKPOINT   = Path("/kaggle/input/cnn-results/best_cnn.pth")
LSTM_CHECKPOINT  = Path("/kaggle/input/lstm-results/best_lstm.pth")
CNN_PREDICTIONS  = Path("/kaggle/input/cnn-results/cnn_predictions.csv")
LSTM_PREDICTIONS = Path("/kaggle/input/lstm-results/lstm_predictions.csv")
FIGURES_DIR      = Path("/kaggle/working/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# --- Device ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

## 1.1 Reproducibility

In [ ]:
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Random seed set to {SEED}")

## 2. Load Predictions

Load pre-computed CNN and LSTM predictions from disk and validate consistency.

In [ ]:
cnn_df  = pd.read_csv(CNN_PREDICTIONS)
lstm_df = pd.read_csv(LSTM_PREDICTIONS)

required_cols = {"y_true", "y_pred", "true_genre", "pred_genre"}
assert required_cols.issubset(cnn_df.columns),  f"CNN predictions missing columns: {required_cols - set(cnn_df.columns)}"
assert required_cols.issubset(lstm_df.columns), f"LSTM predictions missing columns: {required_cols - set(lstm_df.columns)}"

print(f"CNN predictions shape:  {cnn_df.shape}")
print(f"LSTM predictions shape: {lstm_df.shape}")
print("\nCNN (first 3 rows):")
print(cnn_df.head(3).to_string(index=False))
print("\nLSTM (first 3 rows):")
print(lstm_df.head(3).to_string(index=False))

assert len(cnn_df) == len(lstm_df), "CNN and LSTM prediction files have different number of rows!"
assert (cnn_df["y_true"].values == lstm_df["y_true"].values).all(), "y_true values differ between CNN and LSTM!"
print(f"\nBoth files have {len(cnn_df):,} samples with identical y_true ordering.")

## 3. Load Genre Mapping

Parse `genre_mapping.json` and extract the ordered list of class names.

In [ ]:
mapping_path = TENSORS_DIR / "genre_mapping.json"
with open(mapping_path) as f:
    raw_mapping = json.load(f)

first_key = next(iter(raw_mapping))
if isinstance(raw_mapping[first_key], int):
    # Format: {"Rock": 0, "Pop": 1, ...}
    idx_to_genre = {v: k for k, v in raw_mapping.items()}
else:
    # Format: {"0": "Rock", "1": "Pop", ...}
    idx_to_genre = {int(k): v for k, v in raw_mapping.items()}

class_names = [idx_to_genre[i] for i in sorted(idx_to_genre)]
print(f"Num classes: {len(class_names)}")
print(f"Genres: {class_names}")

## 4. CNN and LSTM Metrics

Compute classification metrics for both deep learning models from their saved predictions.

In [ ]:
cnn_metrics  = compute_metrics(cnn_df["y_true"].values,  cnn_df["y_pred"].values,  class_names)
lstm_metrics = compute_metrics(lstm_df["y_true"].values, lstm_df["y_pred"].values, class_names)

print("=" * 60)
print("CNN Classification Report")
print("=" * 60)
print(cnn_metrics["classification_report"])

print("=" * 60)
print("BiLSTM Classification Report")
print("=" * 60)
print(lstm_metrics["classification_report"])

## 5. SVM Baseline

Train an RBF-kernel SVM on handcrafted MFCC features (per-frame mean and std pooled across time).

> **Note:** RBF SVM scales quadratically with dataset size and may be slower than CNN/LSTM training on larger datasets.

In [ ]:
import time
from torch.utils.data import DataLoader

def extract_mfcc_features(dataset) -> np.ndarray:
    """Pool MFCC tensors to (N, 80) via per-frame mean and std."""
    loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)
    features = []
    for mfccs, _ in loader:
        mean = mfccs.mean(dim=1).numpy()   # (batch, 40)
        std  = mfccs.std(dim=1).numpy()    # (batch, 40)
        features.append(np.concatenate([mean, std], axis=1))  # (batch, 80)
    return np.concatenate(features, axis=0)

train_mfcc = MFCCDataset(TENSORS_DIR, split="train")
test_mfcc  = MFCCDataset(TENSORS_DIR, split="test")

print("Extracting features...")
X_train = extract_mfcc_features(train_mfcc)
X_test  = extract_mfcc_features(test_mfcc)
y_train = train_mfcc.labels.numpy()
y_test  = test_mfcc.labels.numpy()

assert (cnn_df["y_true"].values == y_test).all(), \
    "SVM test labels do not match CNN/LSTM prediction ordering!"

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape:  {X_test.shape}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\nTraining SVM...")
t0  = time.time()
svm = SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced")
svm.fit(X_train, y_train)
print(f"Training time: {time.time() - t0:.1f}s")

svm_pred = svm.predict(X_test)
svm_metrics = compute_metrics(y_test, svm_pred, class_names)

print("\n" + "=" * 60)
print("SVM Classification Report")
print("=" * 60)
print(svm_metrics["classification_report"])

svm_predictions_path = Path("/kaggle/working/svm_predictions.csv")
pd.DataFrame({
    "y_true":     y_test,
    "y_pred":     svm_pred,
    "true_genre": [class_names[i] for i in y_test],
    "pred_genre": [class_names[i] for i in svm_pred],
}).to_csv(svm_predictions_path, index=False)
print(f"SVM predictions saved to {svm_predictions_path}")

## 6. Comparative Results Table

Side-by-side summary of all three models sorted by Macro F1.

In [ ]:
results_df = pd.DataFrame([
    {
        "Model":       "CNN",
        "Accuracy":    round(cnn_metrics["accuracy"],    4),
        "Weighted F1": round(cnn_metrics["weighted_f1"], 4),
        "Macro F1":    round(cnn_metrics["macro_f1"],    4),
    },
    {
        "Model":       "BiLSTM",
        "Accuracy":    round(lstm_metrics["accuracy"],    4),
        "Weighted F1": round(lstm_metrics["weighted_f1"], 4),
        "Macro F1":    round(lstm_metrics["macro_f1"],    4),
    },
    {
        "Model":       "SVM",
        "Accuracy":    round(svm_metrics["accuracy"],    4),
        "Weighted F1": round(svm_metrics["weighted_f1"], 4),
        "Macro F1":    round(svm_metrics["macro_f1"],    4),
    },
]).sort_values("Macro F1", ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))

comparative_path = Path("/kaggle/working/comparative_results.csv")
results_df.to_csv(comparative_path, index=False)
print(f"\nComparative results saved to {comparative_path}")

## 7. Confusion Matrices

Normalised confusion matrices (row = true class) for all three models.

In [ ]:
plot_confusion_matrix(
    cnn_df["y_true"].values, cnn_df["y_pred"].values, class_names,
    save_path=FIGURES_DIR / "cnn_confusion_matrix.png",
    title="CNN — Confusion Matrix",
)

plot_confusion_matrix(
    lstm_df["y_true"].values, lstm_df["y_pred"].values, class_names,
    save_path=FIGURES_DIR / "lstm_confusion_matrix.png",
    title="BiLSTM — Confusion Matrix",
)

plot_confusion_matrix(
    y_test, svm_pred, class_names,
    save_path=FIGURES_DIR / "svm_confusion_matrix.png",
    title="SVM — Confusion Matrix",
)

## 8. Training Curves

Plot loss and accuracy curves for CNN and BiLSTM from their saved checkpoints.

In [ ]:
for ckpt_path, model_name in [
    (CNN_CHECKPOINT,  "cnn"),
    (LSTM_CHECKPOINT, "lstm"),
]:
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    history = ckpt.get("history", None)
    if history is not None:
        label = "CNN" if model_name == "cnn" else "BiLSTM"
        plot_training_curves(
            history,
            save_path=FIGURES_DIR / f"{model_name}_training_curves.png",
            title=f"{label} — Training Curves",
        )
    else:
        print(f"No history found in {ckpt_path.name}, skipping training curves.")

## 9. Per-Class F1 Comparison

Grouped bar chart showing per-genre F1 scores for all three models.

In [ ]:
per_class_df = pd.DataFrame({
    "Genre": class_names,
    "CNN":   [cnn_metrics["per_class_f1"][g]  for g in class_names],
    "BiLSTM":[lstm_metrics["per_class_f1"][g] for g in class_names],
    "SVM":   [svm_metrics["per_class_f1"][g]  for g in class_names],
})

per_class_long = per_class_df.melt(id_vars="Genre", var_name="Model", value_name="F1")

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=per_class_long, x="Genre", y="F1", hue="Model", ax=ax)
ax.set_title("Per-Class F1 Score — CNN vs BiLSTM vs SVM")
ax.set_xlabel("Genre")
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1)
ax.legend(title="Model")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

save_path = FIGURES_DIR / "per_class_f1_comparison.png"
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

## 10. Summary

Best model by each metric and the 3 weakest genres per model.

In [ ]:
all_metrics = {"CNN": cnn_metrics, "BiLSTM": lstm_metrics, "SVM": svm_metrics}

best_accuracy = max(all_metrics, key=lambda m: all_metrics[m]["accuracy"])
best_macro_f1 = max(all_metrics, key=lambda m: all_metrics[m]["macro_f1"])

print(f"Best model by accuracy:  {best_accuracy} ({all_metrics[best_accuracy]['accuracy']:.4f})")
print(f"Best model by macro F1:  {best_macro_f1} ({all_metrics[best_macro_f1]['macro_f1']:.4f})")

weakest_genres = {}
print()
for model_name, metrics in all_metrics.items():
    sorted_genres = sorted(metrics["per_class_f1"].items(), key=lambda x: x[1])
    weakest = sorted_genres[:3]
    weakest_genres[model_name] = [{"genre": g, "f1": round(f, 4)} for g, f in weakest]
    print(f"{model_name} weakest genres:")
    for genre, f1 in weakest:
        print(f"  {genre}: {f1:.4f}")

final_results = {
    "cnn_metrics":          {k: v for k, v in cnn_metrics.items()  if k != "classification_report"},
    "lstm_metrics":         {k: v for k, v in lstm_metrics.items() if k != "classification_report"},
    "svm_metrics":          {k: v for k, v in svm_metrics.items()  if k != "classification_report"},
    "comparative_results":  results_df.to_dict(orient="records"),
    "weakest_genres":       weakest_genres,
}

final_results_path = Path("/kaggle/working/final_results.json")
with open(final_results_path, "w") as f:
    json.dump(final_results, f, indent=2)
print(f"\nFinal results saved to {final_results_path}")

## 11. Output Files

List all generated files with sizes.

In [ ]:
output_files = [
    comparative_path,
    final_results_path,
    svm_predictions_path,
    *sorted(FIGURES_DIR.glob("*.png")),
]

print("Output files:")
for p in output_files:
    size_kb = p.stat().st_size / 1024 if p.exists() else 0
    print(f"  {p}  ({size_kb:.1f} KB)")